# Notebook 06: Causal Inference from Observational Data

## Research question

How much can selection bias distort the estimated email effect, and how effectively do standard observational estimators recover the randomized benchmark?

## Analytical objective

Construct an observational analogue with known selection bias, estimate treatment effects naively and with propensity-score methods, and quantify residual error relative to the experimental ATE.

## Statistical framework

The estimators are nearest-neighbor propensity-score matching, inverse probability weighting, and a doubly robust estimator combining a treatment model with an outcome model. Identification is conditional on measured confounders, correct model specification, and positivity.

## Required output

Complete the TODO cells, document the induced imbalance, report ATEs and percentage error by method, diagnose propensity-score overlap, and explain why residual bias remains.

## Scope limitation

This is a controlled simulation using the randomized experiment as ground truth. It demonstrates estimator behavior; it does not establish that the same methods identify causal effects in an arbitrary observational dataset.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import ttest_ind
from scipy.special import expit  # sigmoid
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
import statsmodels.formula.api as smf

np.random.seed(42)

In [2]:
hillstrom_df = pd.read_csv("../data/raw/hillstrom.csv")

display(hillstrom_df.head(), hillstrom_df.tail())

display(hillstrom_df.info(), hillstrom_df.describe())

,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
63995,10,2) $100 - $200,105.54,1,0,Urban,0,Web,Mens E-Mail,0,0,0.0
63996,5,1) $0 - $100,38.91,0,1,Urban,1,Phone,Mens E-Mail,0,0,0.0
63997,6,1) $0 - $100,29.99,1,0,Urban,1,Phone,Mens E-Mail,0,0,0.0
63998,1,5) $500 - $750,552.94,1,0,Surburban,1,Multichannel,Womens E-Mail,0,0,0.0
63999,1,4) $350 - $500,472.82,0,1,Surburban,0,Web,Mens E-Mail,0,0,0.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64000 entries, 0 to 63999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   recency          64000 non-null  int64  
 1   history_segment  64000 non-null  object 
 2   history          64000 non-null  float64
 3   mens             64000 non-null  int64  
 4   womens           64000 non-null  int64  
 5   zip_code         64000 non-null  object 
 6   newbie           64000 non-null  int64  
 7   channel          64000 non-null  object 
 8   segment          64000 non-null  object 
 9   visit            64000 non-null  int64  
 10  conversion       64000 non-null  int64  
 11  spend            64000 non-null  float64
dtypes: float64(2), int64(6), object(4)
memory usage: 5.9+ MB


None

,recency,history,mens,womens,newbie,visit,conversion,spend
count,64000.000000,64000.000000,64000.000000,64000.000000,64000.000000,64000.000000,64000.000000,64000.000000
mean,5.763734,242.085656,0.551031,0.549719,0.502250,0.146781,0.009031,1.050908
std,3.507592,256.158608,0.497393,0.497526,0.499999,0.353890,0.094604,15.036448
min,1.000000,29.990000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.000000,64.660000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,6.000000,158.110000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000
75%,9.000000,325.657500,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000
max,12.000000,3345.930000,1.000000,1.000000,1.000000,1.000000,1.000000,499.000000


## Establish the experimental ATE

We first calculate the Mens-vs-Control effect from the randomized data. This is the benchmark against which every biased-data estimator will be compared.

In [13]:
mens_control_df = hillstrom_df[hillstrom_df["segment"].isin(['Mens E-Mail', 'No E-Mail'])]

In [14]:
# Initialize an empty DataFrame to store results
results_df = pd.DataFrame(columns=["method", "visit", "conversion", "spend"])

In [15]:
# TODO: Estimate the experimental ATE for visit and conversion with
# two-proportion z-tests. Save the effects in `exp_ates`.
mens_control_df = mens_control_df.copy()
mens_control_df["treatment"] = (mens_control_df["segment"] == "Mens E-Mail").astype(int)

exp_ates = {}

for outcome in ["visit", "conversion"]:
    treated = mens_control_df.loc[mens_control_df["treatment"] == 1, outcome]
    control = mens_control_df.loc[mens_control_df["treatment"] == 0, outcome]

    counts = [treated.sum(), control.sum()]
    nobs = [treated.count(), control.count()]
    z_stat, p_value = proportions_ztest(counts, nobs)

    exp_ates[outcome] = treated.mean() - control.mean()

    print(
        f"{outcome}: ATE={exp_ates[outcome]:.4f}, "
        f"z={z_stat:.3f}, p-value={p_value:.3e}"
    )

exp_ates

visit: ATE=0.0766, z=22.486, p-value=5.685e-112
conversion: ATE=0.0068, z=7.385, p-value=1.523e-13


{'visit': 0.07658956365153125, 'conversion': 0.006805006519615695}

In [16]:
# TODO: Estimate the experimental ATE for spend with Welch's t-test.
# Append the complete experimental ground-truth row to `results_df`.
treated_spend = mens_control_df.loc[mens_control_df["treatment"] == 1, "spend"]
control_spend = mens_control_df.loc[mens_control_df["treatment"] == 0, "spend"]

t_stat, p_value = ttest_ind(treated_spend, control_spend, equal_var=False)
exp_ates["spend"] = treated_spend.mean() - control_spend.mean()

print(
    f"spend: ATE={exp_ates['spend']:.4f}, "
    f"t={t_stat:.3f}, p-value={p_value:.3e}"
)

experimental_method = "Experimental (ground truth)"
results_df = results_df.loc[results_df["method"] != experimental_method].copy()
experimental_row = pd.DataFrame([
    {"method": experimental_method, **exp_ates}
])
results_df = (
    experimental_row
    if results_df.empty
    else pd.concat([results_df, experimental_row], ignore_index=True)
)

results_df

spend: ATE=0.7698, t=5.300, p-value=1.164e-07


## Introduce artificial selection bias

The simulation keeps every control customer but retains treated customers with probability `expit(-0.5 * recency_std + 0.5 * history_std)`. Lower recency and higher history therefore increase treatment representation. This creates a known confounding mechanism while preserving the original outcome values.

In [ ]:
# TODO: Create a biased observational sample.
# Keep all controls, but retain treated customers with probability
# expit(-0.5 * recency_std + 0.5 * history_std). Compare covariate means before
# and after selection to verify that confounding was introduced.
mens_control_df["recency_std"] = (
    mens_control_df["recency"] - mens_control_df["recency"].mean()
) / mens_control_df["recency"].std()
mens_control_df["history_std"] = (
    mens_control_df["history"] - mens_control_df["history"].mean()
) / mens_control_df["history"].std()

mens_control_df["selection_probability"] = expit(
    -0.5 * mens_control_df["recency_std"]
    + 0.5 * mens_control_df["history_std"]
)

rng = np.random.default_rng(42)
treated_mask = mens_control_df["treatment"] == 1
retained_treated = treated_mask & (
    rng.random(len(mens_control_df))
    < mens_control_df["selection_probability"]
)
keep_mask = (~treated_mask) | retained_treated
biased_df = mens_control_df.loc[keep_mask].copy()

print(f"Original sample size: {len(mens_control_df):,}")
print(f"Biased sample size: {len(biased_df):,}")
print(f"Treated customers retained: {retained_treated.sum():,} / {treated_mask.sum():,}")

### Bias check
- Check the mean recency and history between treatment and control in both the original and biased datasets. In the original they should be balanced (close to equal), in the biased dataset the treated group should have lower recency and higher history than control.

In [ ]:
for df, label in [(mens_control_df, "Original"), (biased_df, "Biased")]:
    print(label)
    display(df.groupby("treatment")[["recency", "history"]].mean())

## ATE after artificial selection bias

In [ ]:
# TODO: Re-estimate the naive ATEs on `biased_df` for visit and conversion.
naive_ates = {}

for outcome in ["visit", "conversion"]:
    treated = biased_df.loc[biased_df["treatment"] == 1, outcome]
    control = biased_df.loc[biased_df["treatment"] == 0, outcome]

    counts = [treated.sum(), control.sum()]
    nobs = [treated.count(), control.count()]
    z_stat, p_value = proportions_ztest(counts, nobs)

    naive_ates[outcome] = treated.mean() - control.mean()

    print(
        f"{outcome}: ATE={naive_ates[outcome]:.4f}, "
        f"z={z_stat:.3f}, p-value={p_value:.3e}"
    )

naive_ates

In [ ]:
# TODO: Calculate the naive biased spend ATE and append it to results_df.
treated_spend = biased_df.loc[biased_df["treatment"] == 1, "spend"]
control_spend = biased_df.loc[biased_df["treatment"] == 0, "spend"]

t_stat, p_value = ttest_ind(treated_spend, control_spend, equal_var=False)
naive_ates["spend"] = treated_spend.mean() - control_spend.mean()

print(
    f"spend: ATE={naive_ates['spend']:.4f}, "
    f"t={t_stat:.3f}, p-value={p_value:.3e}"
)

naive_method = "Naive biased"
results_df = results_df.loc[results_df["method"] != naive_method].copy()
results_df = pd.concat(
    [
        results_df,
        pd.DataFrame([{"method": naive_method, **naive_ates}]),
    ],
    ignore_index=True,
)

results_df

## Method 1: propensity score matching

A logistic regression estimates `P(Treatment = 1 | recency, history)`. Each treated customer is then paired with the closest control on that score. We check standardized mean differences before and after matching; balance is a diagnostic, not a guarantee of causal identification.

In [ ]:
# TODO: Estimate propensity scores with logistic regression using recency
# and history, match each treated unit to its nearest control, calculate PSM
# ATEs, and check SMD before/after matching.
X = biased_df[["recency", "history"]]
treatment = biased_df["treatment"]

propensity_model = LogisticRegression(C=1e6, max_iter=1000)
propensity_model.fit(X, treatment)
biased_df["propensity_score"] = propensity_model.predict_proba(X)[:, 1]

treated_df = biased_df.loc[biased_df["treatment"] == 1].copy()
control_df = biased_df.loc[biased_df["treatment"] == 0].copy()

matcher = NearestNeighbors(n_neighbors=1)
matcher.fit(control_df[["propensity_score"]])
match_distances, match_indices = matcher.kneighbors(
    treated_df[["propensity_score"]]
)

matched_controls = control_df.iloc[match_indices.ravel()].copy()
matched_df = pd.concat([treated_df, matched_controls], ignore_index=True)

psm_ates = {"method": "Propensity score matching"}
for outcome in ["visit", "conversion", "spend"]:
    psm_ates[outcome] = (
        treated_df[outcome].mean() - matched_controls[outcome].mean()
    )

results_df = results_df.loc[
    results_df["method"] != psm_ates["method"]
].copy()
results_df = pd.concat(
    [results_df, pd.DataFrame([psm_ates])], ignore_index=True
)

pd.DataFrame([psm_ates])

In [ ]:
# Check balancing

def smd(group1, group2):
    pooled_sd = np.sqrt((group1.var() + group2.var()) / 2)
    return abs(group1.mean() - group2.mean()) / pooled_sd

print("Standardized Mean Differences after matching (< 0.1 indicates good balance):")
for col in ["recency", "history"]:
    before = smd(biased_df[biased_df["treatment"] == 1][col], biased_df[biased_df["treatment"] == 0][col])
    after = smd(matched_df[matched_df["treatment"] == 1][col], matched_df[matched_df["treatment"] == 0][col])
    print(f"  {col}: before={before:.3f}, after={after:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(biased_df[biased_df["treatment"] == 0]["propensity_score"], bins=30, alpha=0.5, label="Control")
ax.hist(biased_df[biased_df["treatment"] == 1]["propensity_score"], bins=30, alpha=0.5, label="Treated")

ax.set_xlabel("Propensity Score")
ax.set_ylabel("Count")
ax.set_title("Propensity Score Distribution by Treatment Group")
ax.legend()
plt.tight_layout()
plt.show()


## Method 2: inverse probability weighting

IPW gives each customer a weight inverse to the probability of receiving the treatment they actually received. Under-represented types receive more weight so the weighted groups resemble a common target population. Extreme propensity scores can create unstable weights, so the score range is inspected first.

In [ ]:
# TODO: Construct IPW weights: 1/e for treated and 1/(1-e) for controls.
# Compute weighted ATEs for visit, conversion, and spend. Inspect the score
# range for positivity and extreme-weight problems.
e = biased_df["propensity_score"].clip(1e-6, 1 - 1e-6)
biased_df["ipw"] = np.where(
    biased_df["treatment"] == 1,
    1 / e,
    1 / (1 - e),
)

ipw_ates = {"method": "IPW"}
treated_df = biased_df.loc[biased_df["treatment"] == 1]
control_df = biased_df.loc[biased_df["treatment"] == 0]

for outcome in ["visit", "conversion", "spend"]:
    treated_mean = np.average(
        treated_df[outcome], weights=treated_df["ipw"]
    )
    control_mean = np.average(
        control_df[outcome], weights=control_df["ipw"]
    )
    ipw_ates[outcome] = treated_mean - control_mean

results_df = results_df.loc[results_df["method"] != ipw_ates["method"]].copy()
results_df = pd.concat(
    [results_df, pd.DataFrame([ipw_ates])], ignore_index=True
)

display(pd.DataFrame([ipw_ates]))
biased_df[["propensity_score", "ipw"]].describe()

In [ ]:
biased_df["propensity_score"].describe()

### IPW diagnostics
- The estimated propensity scores range from roughly 0.20 to 0.82, so neither treatment arm has probabilities close to zero throughout the observed sample. This supports practical overlap and avoids severely explosive IPW weights.
- The score standard deviation is about 0.08. Although the distributions overlap, treatment membership is still associated with recency and history, as shown by the pre-adjustment SMDs.
- Overlap is necessary but not sufficient: balance after adjustment and sensitivity to the propensity-model specification must also be checked.

## Method 3: doubly robust estimation

Doubly robust estimation combines a treatment model (the propensity score) with an outcome model. Its protection is that consistency can survive misspecification of one model if the other is correctly specified—but it still depends on measured covariates and overlap.

In [ ]:
# TODO: For each outcome, fit an outcome model with treatment, recency,
# and history. Combine model predictions with the propensity-score residual
# correction to calculate the doubly robust ATE.
dr_ates = {"method": "Doubly robust"}
e = biased_df["propensity_score"].clip(1e-6, 1 - 1e-6)
treatment = biased_df["treatment"]

for outcome in ["visit", "conversion", "spend"]:
    outcome_model = smf.ols(
        f"{outcome} ~ treatment + recency + history",
        data=biased_df,
    ).fit()

    predict_treated = biased_df.copy()
    predict_treated["treatment"] = 1
    predict_control = biased_df.copy()
    predict_control["treatment"] = 0

    mu_1 = outcome_model.predict(predict_treated)
    mu_0 = outcome_model.predict(predict_control)
    observed = biased_df[outcome]

    aipw_score = (
        mu_1
        - mu_0
        + treatment * (observed - mu_1) / e
        - (1 - treatment) * (observed - mu_0) / (1 - e)
    )
    dr_ates[outcome] = aipw_score.mean()

results_df = results_df.loc[results_df["method"] != dr_ates["method"]].copy()
results_df = pd.concat(
    [results_df, pd.DataFrame([dr_ates])], ignore_index=True
)

pd.DataFrame([dr_ates])

## Compare all estimators with the experimental benchmark

The final table reports each estimated ATE and its percentage difference from the randomized ground truth. This makes the lesson concrete: adjustment reduces bias, but the remaining error reveals the limits of the two available confounders.

In [ ]:
results_df.set_index("method").round(4)

In [ ]:
# TODO: Compare every estimator with the experimental ground truth.
# Report both the ATE table and percentage error relative to the ground truth.
outcomes = ["visit", "conversion", "spend"]
ate_table = results_df.set_index("method")[outcomes].astype(float)
ground_truth = ate_table.loc["Experimental (ground truth)"]

percentage_error = (
    ate_table.drop(index="Experimental (ground truth)")
    .subtract(ground_truth)
    .divide(ground_truth)
    .multiply(100)
    .round(1)
)
percentage_error.columns = [
    "Visit % error",
    "Conversion % error",
    "Spend % error",
]

print("ATE estimates")
display(ate_table.round(4))
print("Signed percentage error relative to experimental ground truth")
percentage_error

## Causal-inference takeaway

The artificial selection mechanism breaks the covariate balance created by randomization: retained treated customers have lower recency and higher historical spend than controls. Consequently, the naive comparison overstates the randomized ATE by approximately 14.4% for visit, 17.0% for conversion, and 11.8% for spend.

No adjustment method dominates every outcome. IPW is closest for visit (-1.3% error) and spend (-9.7%), while propensity-score matching is closest for conversion (+2.0%). Matching achieves good observed balance but still understates visit by 6.7% and overstates spend by 15.8%. The doubly robust estimates are close to IPW, with errors of -1.5%, -4.7%, and -10.0% for visit, conversion, and spend respectively.

In this controlled simulation, selection was generated only from the two observed covariates, so the residual error is more naturally attributed to finite-sample variation, propensity/outcome-model misspecification, and imperfect nearest-neighbor matching than to genuine unmeasured confounding. In a real observational study, omitted variables such as channel, product affinity, customer tenure, geography, and new-customer status could also violate conditional exchangeability. Adjustment reduces known selection bias here, but does not by itself prove causal identification in a new dataset.